# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the FAIR principles for scientific data.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using croissant
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we enumerate all available record sets (tables) in the dataset, along with their associated fields (columns).

**Note:** All references use the schema `@id` fields for precise and future-proof access.

In [ ]:
# List all record sets and their fields using their @id
record_sets = []
for record_set in dataset.metadata.record_sets:
    print(f"Record Set: {record_set.name} (id: {record_set['@id']})")
    record_sets.append(record_set['@id'])
    for field in record_set.fields:
        print(f" |- Field: {field.name} (id: {field['@id']}, type: {field.data_type})")

if not record_sets:
    print("No explicit record sets found in this dataset's metadata.")
    print("If this is the case, let's try to infer record sets from file objects.")

    # Try to enumerate file objects
    for fobj in dataset.metadata.file_objects:
        print(f"FileObject: {fobj.name} (id: {fobj['@id']})")
        if hasattr(fobj, 'columns'):
            for col in fobj.columns:
                print(f" |- Column: {col.name} (id: {col['@id']}, type: {col.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> ⚠️ **Note:** All entities are referenced by their `@id` as shown above. Adjust `record_set_id` below as needed.

If record sets were not found explicitly above, but file object(s) were shown, you can provide the file object's `@id` to `dataset.records()`.

In [ ]:
# If available, use identified record_set_ids or fallback to file_object id(s)
# Example: Suppose we found one primary file object with @id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
record_set_ids = [
    'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}")
    print(f"Fields: {df.columns.tolist()}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing, and grouping using numeric and categorical fields.

Choose relevant fields via their `@id` (column name) for EDA. Here, we use example identifiers.

*Identify fields/columns via their `@id` names printed above, e.g., `MW`, `Abundance`, `pI`, etc., if present.*

In [ ]:
# Pick the main table/file object @id
record_set_id = 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034'
df = dataframes[record_set_id]

# Choose a numeric field (column) by its @id (or name) -- e.g., 'MW' for Molecular Weight
# If not sure, print df.columns
print(df.columns.tolist())

# Example: Assume 'MW' (molecular weight) and 'Abundance' as numeric fields, 'description' as group field
numeric_field = 'MW'  # <-- replace with the correct field name if different
group_field = 'description'  # <-- or another categorical field

# EDA: Filter, normalize, group
if numeric_field in df.columns:
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)})")
    display(filtered_df[[numeric_field]].head())
    
    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print(f"Numeric field '{numeric_field}' not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize distributions and relationships using your selected fields. Update the code below if your field names differ.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If normalized field calculated, plot that too
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=40, kde=True)
        plt.title(f"Normalized {numeric_field} Distribution (Filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel("Count")
        plt.show()

    # Example: scatter plot
    if 'Abundance' in df.columns:
        plt.figure(figsize=(8,6))
        sns.scatterplot(data=df, x=numeric_field, y='Abundance')
        plt.title(f"{numeric_field} vs. Abundance")
        plt.xlabel(numeric_field)
        plt.ylabel('Abundance')
        plt.show()


## 6. Conclusion
In this notebook, you've learned how to load a FAIR² dataset via its Croissant schema using `mlcroissant`,explored its metadata, extracted structured tables by their `@id`, and performed preliminary analysis and visualization.
Key takeaways from your EDA:
- Loaded molecular protein data identified via mass spectrometry.
- Explored distributions for fields such as molecular weight (`MW`) and `Abundance`.
- Demonstrated normalization, filtering, and basic grouping for quantitative insight.

You are now set up to pursue further analysis for biological or computational research!